# Data Collection — Fama-French Five-Factor Model
**Russian stock market, 2014–2026**

The notebook collects all input data for analysis in R. Run once; results are saved to CSV.

| File | Description |
|---|---|
| `monthly_prices.csv` | Monthly closing prices (wide: dates × tickers) |
| `mkt_factor.csv` | Market factor MKT (IMOEX − RF) |
| `rf_monthly.csv` | Risk-free rate of the Central Bank of the Russian Federation, monthly effective |
| `fundament.csv` | Fundamentals (long: date, ticker, metrics) |

## Library imports

In [ ]:
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import numpy as np
from pathlib import Path
from functools import reduce

## 1. Stock Prices
Download daily closing prices from MOEX ISS for all tickers in the sample, aggregate them to monthly prices (last price of the month), and save them in wide format.

### 1.1 Downloading daily prices from MOEX ISS

In [ ]:
# итоговая отфильтрованная выборка по 
TICKERS = [
    "ABRD","AFKS","AFLT","AKRN","ALRS","APTK","AQUA","ARSA","ASSB","BANE","BANEP","BISVP",
    "BLNG","BRZL","BSPB","CBOM","CHMF","CHMK","CNTL","CNTLP","DIOD","DVEC","DZRD","DZRDP",
    "FEES","FESH","GAZP","GCHE","GMKN","HIMCP","HYDR","IGST","IGSTP","IRAO","IRKT","JNOS",
    "JNOSP","KAZT","KAZTP","KBSB","KCHE","KLSB","KMAZ","KMEZ","KOGK","KRKNP","KRKOP","KROT",
    "KROTP","KRSB","KRSBP","KUZB","KZOS","KZOSP","LIFE","LKOH","LSNG","LSNGP","LSRG","LVHK",
    "MAGE","MAGEP","MAGN","MFGS","MFGSP","MGNT","MGTS","MGTSP","MOEX","MRKC","MRKK","MRKP",
    "MRKS","MRKU","MRKV","MRKY","MRKZ","MRSB","MSNG","MSRS","MSTT","MTLR","MTLRP","MTSS",
    "MVID","NAUK","NFAZ","NKHP","NKNC","NKNCP","NKSH","NLMK","NMTP","NNSBP","NSVZ","NVTK",
    "OGKB","OMZZP","PAZA","PHOR","PIKK","PLZL","PMSB","PMSBP","RASP","RBCM","RGSS","RKKE",
    "RNFT","ROLO","ROSN","ROST","RTGZ","RTKM","RTKMP","RTSBP","RUAL","RZSB","SAGO","SAGOP",
    "SARE","SAREP","SBER","SBERP","SELG","SIBN","SNGS","SNGSP","STSB","STSBP","SVAV","TASB",
    "TASBP","TATN","TATNP","TGKA","TGKB","TGKBP","TGKN","TNSE","TORS","TORSP","TRMK","TRNFP",
    "TTLK","TUZA","UKUZ","UNAC","UNKL","UPRO","URKZ","USBN","UTAR","VGSB","VGSBP","VJGZP",
    "VLHZ","VSMO","VSYD","VTBR","WTCM","WTCMP","YAKG","YKEN","YKENP","ZILL","ZVEZ",
]

START_DATE = "2014-06-09"
END_DATE   = "2026-12-31"

# сессия с retry
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1)))

# функция для загрузки одного тикера
def fetch(ticker):
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/{ticker}.json"
    all_rows = []
    start = 0
    while True:
        params = {
            "from": START_DATE,
            "till": END_DATE,
            "start": start,
            "iss.meta": "off",
            "iss.only": "history",
            "history.columns": "TRADEDATE,SECID,CLOSE",
        }
        r = session.get(url, params=params, timeout=30)
        r.raise_for_status()
        rows = r.json()["history"]["data"]
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 100:
            break
        start += 100
        time.sleep(0.1)
    if not all_rows:
        return None
    df = pd.DataFrame(all_rows, columns=["TRADEDATE", "SECID", "CLOSE"])
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
    df["CLOSE"]     = pd.to_numeric(df["CLOSE"], errors="coerce")
    return df

# загрузка всех тикеров
all_data = []
for t in TICKERS:
    print("→", t)
    df = fetch(t)
    if df is not None:
        all_data.append(df)
    time.sleep(0.1)

# сборка в wide-формат и сохранение
df_wide = pd.concat(all_data).pivot(index="TRADEDATE", columns="SECID", values="CLOSE")
df_wide.to_csv("final_wide.csv")

# проверка
print(df_wide.iloc[:, :5].head())
print(df_wide.iloc[:, -5:].tail())

### 1.2 Aggregation to monthly prices

In [1]:
# загружаем дневные цены
df = pd.read_csv("final_wide.csv")
df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
df = df.set_index("TRADEDATE")
df = df.apply(pd.to_numeric, errors="coerce")

# берём последнюю доступную цену в каждом месяце
monthly_prices = df.resample("ME").last()

# сохраняем
monthly_prices.to_csv("monthly_prices.csv")

# проверка
print("Месячные цены:", monthly_prices.shape)
print(monthly_prices.head())

Месячные цены: (144, 167)
              ABRD    AFKS   AFLT    AKRN  ALRS   APTK  AQUA    ARSA    ASSB  \
TRADEDATE                                                                      
2014-06-30  135.88  45.600  56.43  1235.0  41.8  14.23   NaN  0.8723  0.2000   
2014-07-31  136.10  39.400  50.55  1137.4  44.1  12.40   NaN  0.8015  0.2209   
2014-08-31  127.13  37.719  46.73  1144.1  42.5  12.00   NaN  0.8140  0.2670   
2014-09-30  115.01  13.130  43.25  1250.0  35.5  14.15   NaN  0.8178  0.2666   
2014-10-31  111.00  15.950  38.20  1344.0  38.5  14.30   NaN  0.6850  0.2600   

              BANE  ...    VSMO  VSYD     VTBR    WTCM  WTCMP  YAKG    YKEN  \
TRADEDATE           ...                                                       
2014-06-30  2400.0  ...  7400.0   NaN  0.04110  10.950   5.00  11.0  0.1400   
2014-07-31  1963.0  ...  7501.0   NaN  0.03980  14.000   6.50  14.0  0.1417   
2014-08-31  1943.8  ...  7550.0   NaN  0.03840  13.400   6.20  13.2  0.1161   
2014-09-30  1255.0

## 2. Risk-free rate
Load the Central Bank of the Russian Federation's key rate from an Excel file, convert the annual rate to a monthly effective rate, and save.

In [11]:
# загружаем файл ЦБ
rf = pd.read_excel("/Users/prokofiev/Yandex.Disk.localized/Finance/Fama-French model/ЦБ РФ/Инфляция_и_ключевая_ставка_Банка_России_F09_06_2014_T01_04_2026.xlsx")

# оставляем только нужные колонки
rf = rf[["Дата", "Ключевая ставка, % годовых"]].copy()

# преобразуем дату формата '2.2026' -> месяц и год
rf["Дата"] = rf["Дата"].astype(str).str.strip()
rf[["month", "year"]] = rf["Дата"].str.split(".", expand=True)
rf["month"] = pd.to_numeric(rf["month"], errors="coerce")
rf["year"]  = pd.to_numeric(rf["year"],  errors="coerce")

# создаём дату: последний день месяца (для совпадения с monthly_prices)
rf["Date"] = pd.to_datetime(dict(year=rf["year"], month=rf["month"], day=1), errors="coerce")
rf["Date"] = rf["Date"] + pd.offsets.MonthEnd(0)

# приводим ставку к числу и переводим в доли
rf["Rate"] = pd.to_numeric(rf["Ключевая ставка, % годовых"], errors="coerce") / 100

# убираем мусор и сортируем
rf = rf[["Date", "Rate"]].dropna().sort_values("Date").reset_index(drop=True)

# переводим годовую ставку в месячную эффективную: (1 + r)^(1/12) - 1
rf["RF_month"] = (1 + rf["Rate"]) ** (1 / 12) - 1

# итоговая серия
rf_monthly = rf.set_index("Date")["RF_month"]

# вручную добавляем три месяца 2020 года, отсутствующие в исходном файле ЦБ
missing_2020 = pd.Series({
    pd.Timestamp("2020-01-31"): 0.00506,
    pd.Timestamp("2020-10-31"): 0.00347,
    pd.Timestamp("2020-11-30"): 0.00347,
    pd.Timestamp("2020-12-31"): 0.00347,
}, name="RF_month")
rf_monthly = pd.concat([rf_monthly, missing_2020]).sort_index()
rf_monthly.index.name = "Date"

# сохраняем
rf_monthly.to_csv("rf_monthly.csv")

# проверка
print(rf_monthly.head(12))
print(rf_monthly.tail(12))

Date
2014-06-30    0.006045
2014-07-31    0.006434
2014-08-31    0.006434
2014-09-30    0.006434
2014-10-31    0.006434
2014-11-30    0.007592
2014-12-31    0.013170
2015-01-31    0.013170
2015-02-28    0.011715
2015-03-31    0.010979
2015-04-30    0.010979
2015-05-31    0.009864
Name: RF_month, dtype: float64
Date
2025-03-31    0.016012
2025-04-30    0.016012
2025-05-31    0.016012
2025-06-30    0.015309
2025-07-31    0.013888
2025-08-31    0.013888
2025-09-30    0.013170
2025-10-31    0.012808
2025-11-30    0.012808
2025-12-31    0.012445
2026-01-31    0.012445
2026-02-28    0.012081
Name: RF_month, dtype: float64


/Users/prokofiev/miniconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


## 3. Market Factor (MKT)
Download daily IMOEX index values from MOEX ISS, aggregate them to monthly returns, and subtract the risk-free rate.

In [12]:
# загружаем дневной индекс IMOEX с MOEX ISS
def fetch_index():
    url = "https://iss.moex.com/iss/history/engines/stock/markets/index/boards/SNDX/securities/IMOEX.json"
    all_rows = []
    start = 0
    while True:
        params = {
            "from": "2014-06-09",
            "till": "2026-05-29",
            "start": start,
            "iss.meta": "off",
            "iss.only": "history",
            "history.columns": "TRADEDATE,SECID,CLOSE",
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        rows = r.json()["history"]["data"]
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < 100:
            break
        start += 100
        time.sleep(0.1)
    df = pd.DataFrame(all_rows, columns=["TRADEDATE", "SECID", "CLOSE"])
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"], errors="coerce")
    df["CLOSE"]     = pd.to_numeric(df["CLOSE"],     errors="coerce")
    return df

df_index = fetch_index()

# берём последнее значение индекса в каждом месяце
monthly_index = df_index.set_index("TRADEDATE").resample("ME").last()

# считаем месячные доходности индекса
index_returns = monthly_index["CLOSE"].pct_change().dropna()

# загружаем безрисковую ставку
rf = pd.read_csv("rf_monthly.csv")
rf["Date"] = pd.to_datetime(rf["Date"])
rf = rf.set_index("Date").rename(columns={rf.columns[0]: "RF_month"})

# считаем рыночный фактор: MKT = доходность индекса - RF
mkt = pd.DataFrame(index_returns).join(rf, how="inner")
mkt["MKT"] = mkt["CLOSE"] - mkt["RF_month"]
mkt_factor = mkt[["MKT"]]

# сохраняем
mkt_factor.to_csv("mkt_factor.csv")

# проверка
print(mkt_factor.head())
print(mkt_factor.tail())

                 MKT
TRADEDATE           
2014-07-31 -0.071979
2014-08-31  0.008860
2014-09-30  0.000962
2014-10-31  0.048418
2014-11-30  0.022782
                 MKT
TRADEDATE           
2025-10-31 -0.072184
2025-11-30  0.047076
2025-12-31  0.021257
2026-01-31 -0.006619
2026-02-28 -0.006187


## 4. Fundamental Data
We collect balance sheet, income, and market indicators from files on the [difan](https://difan.xyz/ru/) website. We calculate the gaps in `marketCapitalization` using December prices and the number of shares in recent years.

### 4.1 Downloading and merging files from the difan website

In [4]:
# путь к папке с файлами difan
DATA_DIR = Path("/Users/prokofiev/Yandex.Disk.localized/Finance/Fama-French model/difan")

# нужные поля из каждого типа файлов
BS_FIELDS = ["date", "symbol", "totalAssets", "totalStockholdersEquity"]
IS_FIELDS = ["date", "symbol", "revenue", "costOfRevenue", "operatingIncome",
             "sellingGeneralAndAdministrativeExpenses", "interestExpense"]
EV_FIELDS = ["date", "symbol", "marketCapitalization", "numberOfShares"]

# файлы difan хранятся в транспонированном виде (строки = поля, столбцы = периоды)
def parse_transposed(path, fields):
    raw = pd.read_excel(path, header=None, index_col=0)
    raw.columns = range(raw.shape[1])
    df = raw.T.reset_index(drop=True)
    df.columns = df.columns.astype(str)
    df = df[fields].copy()
    df["date"]   = pd.to_datetime(df["date"])
    df["symbol"] = df["symbol"].str.replace(".ME", "", regex=False)
    for f in fields:
        if f not in ("date", "symbol"):
            df[f] = pd.to_numeric(df[f], errors="coerce")
    return df

# enterprise value файлы хранятся в обычном виде
def parse_normal(path, fields):
    df = pd.read_excel(path, usecols=fields)
    df["date"]   = pd.to_datetime(df["date"])
    df["symbol"] = df["symbol"].str.replace(".ME", "", regex=False)
    return df

# загружаем все файлы из папки
dfs = {"bs": [], "is": [], "ev": []}
for path in DATA_DIR.glob("*.xlsx"):
    name = path.stem
    try:
        if "balance sheet" in name:
            dfs["bs"].append(parse_transposed(path, BS_FIELDS))
        elif "income statement" in name:
            dfs["is"].append(parse_transposed(path, IS_FIELDS))
        elif "enterprise value" in name:
            dfs["ev"].append(parse_normal(path, EV_FIELDS))
    except Exception as e:
        print(f"skipped {path.name}: {e}")

# объединяем три источника
bs  = pd.concat(dfs["bs"], ignore_index=True)
is_ = pd.concat(dfs["is"], ignore_index=True)
ev  = pd.concat(dfs["ev"], ignore_index=True)

dataset = reduce(lambda l, r: pd.merge(l, r, on=["date", "symbol"], how="outer"), [bs, is_, ev])
dataset = dataset.sort_values(["symbol", "date"]).reset_index(drop=True)

# оставляем только данные с 2014 года
dataset_2014 = dataset[dataset["date"].dt.year >= 2014].copy()

print(f"{dataset_2014.shape[0]} строк, {dataset_2014['symbol'].nunique()} тикеров")

1570 строк, 146 тикеров


### 4.2 Calculating marketCapitalization gaps
For some observations, `marketCapitalization` is missing from the difan. We calculate marketCapitalization gaps using December prices and the number of shares (filling forward and backward by year).

In [5]:
# декабрьские цены из monthly_prices
prices     = pd.read_csv("monthly_prices.csv", parse_dates=["TRADEDATE"])
dec_prices = (prices[prices["TRADEDATE"].dt.month == 12]
              .copy()
              .assign(year=lambda x: x["TRADEDATE"].dt.year)
              .drop(columns=["TRADEDATE"])
              .set_index("year"))

# количество акций: берём из ev, заполняем пропуски вперёд и назад
shares = ev[["symbol", "date", "numberOfShares"]].copy()
shares["year"] = pd.to_datetime(shares["date"]).dt.year
shares = (shares
          .pivot(index="year", columns="symbol", values="numberOfShares")
          .ffill().bfill()
          .reindex(sorted(dec_prices.index.unique()))
          .ffill().bfill())

# расчётная капитализация = цена × количество акций
calc_mc = dec_prices * shares

# заполняем пропуски в marketCapitalization
dataset_2014["year"]   = dataset_2014["date"].dt.year
dataset_2014["ticker"] = dataset_2014["symbol"]

def fill_mc(row):
    if pd.notna(row["marketCapitalization"]):
        return row["marketCapitalization"]
    try:
        return calc_mc.loc[row["year"], row["ticker"]]
    except KeyError:
        return None

dataset_2014["marketCapitalization"] = dataset_2014.apply(fill_mc, axis=1)
dataset_2014 = dataset_2014.drop(columns=["year", "ticker", "numberOfShares"])

# сохраняем
dataset_2014.to_csv("fundament.csv", index=False)

# проверка
print(f"{dataset_2014.shape[0]} строк, {dataset_2014['symbol'].nunique()} тикеров")
print(dataset_2014.head())

1570 строк, 146 тикеров
         date symbol   totalAssets  totalStockholdersEquity       revenue  \
6  2014-12-31   ABRD  9.596832e+09             3.912785e+09  7.733519e+09   
7  2015-12-31   ABRD  1.139809e+10             4.075533e+09  7.824100e+09   
8  2016-12-31   ABRD  1.161214e+10             6.544275e+09  6.935875e+09   
9  2017-12-31   ABRD  1.296930e+10             7.244288e+09  6.628516e+09   
10 2018-12-31   ABRD  1.422564e+10             8.142507e+09  7.660565e+09   

    costOfRevenue  operatingIncome  sellingGeneralAndAdministrativeExpenses  \
6    3.725681e+09     1.417343e+09                             1.425625e+09   
7    3.374431e+09     1.283101e+09                             1.948788e+09   
8    3.087917e+09     1.018499e+09                             1.708907e+09   
9    3.264296e+09     1.365144e+09                             8.672920e+08   
10   3.994509e+09     1.569883e+09                             8.654740e+08   

    interestExpense  marketCapitalizat